In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.cluster import KMeans
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier
)
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC

from xgboost import XGBClassifier

from imblearn.over_sampling import RandomOverSampler

In [2]:
DATA_PATH = Path("../data/diabetes.csv")

df = pd.read_csv(DATA_PATH)

df = df.drop(columns=["Unnamed: 0"])

columns_with_missing_values = [
    "Glucose",
    "BloodPressure",
    "SkinThickness",
    "Insulin",
    "BMI"
]

df[columns_with_missing_values] = df[
    columns_with_missing_values
].replace(0, np.nan)

imputer = SimpleImputer(strategy="median")

df[columns_with_missing_values] = imputer.fit_transform(
    df[columns_with_missing_values]
)

df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age
0,6,148.0,72.0,35.0,125.0,33.6,0.627,50
1,1,85.0,66.0,29.0,125.0,26.6,0.351,31
2,8,183.0,64.0,29.0,125.0,23.3,0.672,32
3,1,89.0,66.0,23.0,94.0,28.1,0.167,21
4,0,137.0,40.0,35.0,168.0,43.1,2.288,33


In [3]:
scaler = StandardScaler()

scaled_features = scaler.fit_transform(df)

scaled_df = pd.DataFrame(
    scaled_features,
    columns=df.columns
)

scaled_df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age
0,0.639947,0.866045,-0.031990,0.670643,-0.181541,0.166619,0.468492,1.425995
1,-0.844885,-1.205066,-0.528319,-0.012301,-0.181541,-0.852200,-0.365061,-0.190672
2,1.233880,2.016662,-0.693761,-0.012301,-0.181541,-1.332500,0.604397,-0.105584
3,-0.844885,-1.073567,-0.528319,-0.695245,-0.540642,-0.633881,-0.920763,-1.041549
4,-1.141852,0.504422,-2.679076,0.670643,0.316566,1.549303,5.484909,-0.020496


In [4]:
kmeans = KMeans(
    n_clusters=4,
    random_state=42,
    n_init=10
)

clusters = kmeans.fit_predict(scaled_df)

df["Cluster"] = clusters

df["Cluster"].value_counts()

Cluster
1    309
2    233
0    184
3     42
Name: count, dtype: int64

In [5]:
X = df.drop("Cluster", axis=1)

y = df["Cluster"]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (768, 8)
y shape: (768,)


In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [7]:
oversampler = RandomOverSampler(random_state=42)

X_train_resampled, y_train_resampled = oversampler.fit_resample(
    X_train,
    y_train
)

print("Original class distribution:")
print(y_train.value_counts())

print("\nResampled class distribution:")
print(pd.Series(y_train_resampled).value_counts())

Original class distribution:
Cluster
1    247
2    186
0    147
3     34
Name: count, dtype: int64

Resampled class distribution:
Cluster
0    247
2    247
1    247
3    247
Name: count, dtype: int64


In [10]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=5000),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42),
    "SVM": SVC(),
    "XGBoost": XGBClassifier(random_state=42)
}

In [11]:
results = []

for model_name, model in models.items():

    model.fit(X_train_resampled, y_train_resampled)

    predictions = model.predict(X_test)

    accuracy = accuracy_score(y_test, predictions)

    precision = precision_score(
        y_test,
        predictions,
        average="weighted"
    )

    recall = recall_score(
        y_test,
        predictions,
        average="weighted"
    )

    f1 = f1_score(
        y_test,
        predictions,
        average="weighted"
    )

    results.append({
        "Model": model_name,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1
    })

    print(f"\n========== {model_name} ==========")

    print(f"Accuracy : {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall   : {recall:.4f}")
    print(f"F1 Score : {f1:.4f}")

/Users/laraibi/projects/machine-learning/diabetes-prediction-and-clustering/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 5000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=5000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(



========== Logistic Regression ==========
Accuracy : 0.9805
Precision: 0.9809
Recall   : 0.9805
F1 Score : 0.9804

========== Decision Tree ==========
Accuracy : 0.8636
Precision: 0.8671
Recall   : 0.8636
F1 Score : 0.8625

========== Random Forest ==========
Accuracy : 0.9156
Precision: 0.9176
Recall   : 0.9156
F1 Score : 0.9154

========== Gradient Boosting ==========
Accuracy : 0.9416
Precision: 0.9425
Recall   : 0.9416
F1 Score : 0.9417

========== SVM ==========
Accuracy : 0.8442
Precision: 0.8476
Recall   : 0.8442
F1 Score : 0.8452

========== XGBoost ==========
Accuracy : 0.9416
Precision: 0.9427
Recall   : 0.9416
F1 Score : 0.9416


In [12]:
results_df = pd.DataFrame(results)

results_df.sort_values(
    by="F1 Score",
    ascending=False
)

,Model,Accuracy,Precision,Recall,F1 Score
0,Logistic Regression,0.980519,0.980925,0.980519,0.980442
3,Gradient Boosting,0.941558,0.942513,0.941558,0.941722
5,XGBoost,0.941558,0.942698,0.941558,0.941645
2,Random Forest,0.915584,0.917619,0.915584,0.915391
1,Decision Tree,0.863636,0.867086,0.863636,0.862499
4,SVM,0.844156,0.847595,0.844156,0.845180
